# Legacy Engine na GPU do Google Colab

1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4 → Salvar**.
2. **Ambiente de execução → Executar tudo** (se avisar que o caderno não é do Google: **Executar mesmo assim**).
3. Espere: a 1ª vez compila o motor (uns 20–40 min). No fim aparecem as imagens e o botão de baixar os resultados.

Os testes ficam em `engine/pedido_colab.txt` no GitHub: para rodar testes novos, basta executar tudo de novo.

In [ ]:
# 1) Vulkan da NVIDIA: o Colab só monta a parte de cálculo do driver; as bibliotecas gráficas (Vulkan) vêm do pacote
#    oficial da NVIDIA da MESMA versão do driver, sem instalar nada no sistema.
import os, subprocess
V = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('Placa:', subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip(), '| driver', V)
script = f'''
set -e
cd /tmp
if [ ! -d /opt/nvvk ]; then
  wget -q https://us.download.nvidia.com/tesla/{V}/NVIDIA-Linux-x86_64-{V}.run || wget -q https://download.nvidia.com/XFree86/Linux-x86_64/{V}/NVIDIA-Linux-x86_64-{V}.run
  sh NVIDIA-Linux-x86_64-{V}.run --extract-only --target nv > /dev/null
  mkdir -p /opt/nvvk && cp nv/*.so.{V} /opt/nvvk/
  cd /opt/nvvk && for f in *.so.{V}; do ln -sf $f ${{f%.{V}}}.1; ln -sf $f ${{f%.{V}}}.0; done
fi
mkdir -p /usr/share/vulkan/icd.d
echo '{{"file_format_version":"1.0.1","ICD":{{"library_path":"/opt/nvvk/libGLX_nvidia.so.{V}","api_version":"1.4.0"}}}}' > /usr/share/vulkan/icd.d/nvidia_icd.json
'''
subprocess.run(['bash', '-c', script], check=True)
os.environ['LD_LIBRARY_PATH'] = '/opt/nvvk:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['XDG_RUNTIME_DIR'] = '/tmp'
os.environ['LEGACY_SEM_CONTA'] = '1'
print('Vulkan da NVIDIA pronto.')

In [ ]:
%%bash
# 2) Ferramentas de compilação e o código do motor (cópia rasa do GitHub, sempre a versão mais nova do main)
apt-get -qq update > /dev/null 2>&1 || true
apt-get -qq install -y vulkan-tools libvulkan-dev glslang-tools spirv-tools ninja-build libx11-dev libxrandr-dev \
  libxinerama-dev libxcursor-dev libxi-dev libxkbcommon-dev libwayland-dev libglfw3-dev > /dev/null 2>&1 || true
vulkaninfo --summary 2>/dev/null | grep -E "deviceName|apiVersion" | head -2
cd /content
if [ -d leagcy ]; then cd leagcy && git fetch -q --depth 1 origin main && git reset -q --hard origin/main; else git clone -q --depth 1 https://github.com/seronzera/leagcy.git; fi
cd /content/leagcy && git log --oneline -1

In [ ]:
%%bash
# 3) Compilar (Release, sem DLSS: o SDK da NVIDIA tem 1,6 GB). A 1ª vez demora; as seguintes só recompilam o que mudou.
cd /content/leagcy/engine
start=$(date +%s)
cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DLEGACY_DLSS=OFF > /tmp/cmake.txt 2>&1 || { tail -30 /tmp/cmake.txt; exit 1; }
cmake --build build --parallel > /tmp/build.txt 2>&1 || { grep -E "error" /tmp/build.txt | head -30; tail -20 /tmp/build.txt; exit 1; }
echo "compilado em $(( ($(date +%s) - start) / 60 )) min"; ls -la build/legacy

In [ ]:
# 4) Rodar os pedidos de engine/pedido_colab.txt (as imagens vão para engine/resultados/)
import subprocess, time, os, shutil
os.chdir('/content/leagcy/engine')
shutil.rmtree('resultados', ignore_errors=True); os.makedirs('resultados')
log = open('resultados/saida.txt', 'w')
for linha in open('pedido_colab.txt', encoding='utf-8'):
    cmd = linha.strip()
    if not cmd or cmd.startswith('#'):
        continue
    print('>', cmd, flush=True); log.write('> ' + cmd + '\n')
    t = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, errors='replace')
    out = p.stdout + p.stderr
    log.write(out + f'\n[{time.time() - t:.0f} s, código {p.returncode}]\n\n'); log.flush()
    for l in out.splitlines():
        if 'GPU ' in l or 'salva' in l or 'erro' in l.lower() or 'error' in l.lower():
            print('   ', l[:300])
    print(f'    [{time.time() - t:.0f} s, código {p.returncode}]', flush=True)
log.close()

In [ ]:
# 5) Imagens e o pacote com tudo (baixe o .zip e mande no chat)
from IPython.display import Image, display
import glob, shutil
for f in sorted(glob.glob('/content/leagcy/engine/resultados/*.png')):
    print(f.split('/')[-1]); display(Image(f, width=960))
shutil.make_archive('/content/resultados_colab', 'zip', '/content/leagcy/engine/resultados')
from google.colab import files
files.download('/content/resultados_colab.zip')